# Chapter 67: Security Log Analysis & SIEM Integration

**Volume 4 — Security Operations**

**The evidence is already in your logs.** Every attack leaves traces across your firewall,
your Active Directory, your DNS resolver, and your proxy. The problem is not missing data —
it is finding the signal inside 10 terabytes of daily noise.
AI does not just filter logs faster. It **understands what the events mean** across sources.

### What You Will Build — 5 Demos

| Demo | Engine | What It Does |
|------|--------|--------------|
| **1. Multi-Source Log Parser** | Regex + normalization | Cisco ASA + Windows Event + CEF → unified schema |
| **2. Pattern-Based Threat Detector** | Rule engine + scoring | Brute force, port scan, firewall deny storms |
| **3. Log Correlation Engine** | Time-window grouping | Links related events across sources by IP + time |
| **4. SIEM Alert Enrichment** | IP reputation + geo | Threat intel overlay on raw alerts |
| **5. Full Log Analysis Pipeline** | Ingest → parse → correlate → enrich → LLM | End-to-end analyst-ready report |

### The Core Insight
> **No single log source is sufficient. Real attacks are multi-stage and multi-source by definition.
> An attack that bypasses network detection may be visible in authentication logs.
> Correlating across all four log categories is what makes attack detection reliable —
> and it is exactly what this pipeline is built to do.**

In [ ]:
# pip install anthropic
import os, json, re, math, time, random
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    return "Simulation: " + prompt[:80] + "..."

print("Setup complete.")

## Demo 1: Multi-Source Log Parser

Every security tool speaks a different dialect. A **Cisco ASA** encodes a firewall deny as:
```
%ASA-4-106023: Deny tcp src outside:185.220.101.47/52111 dst inside:10.0.1.5/443
```
A **Windows Security Event** logs the same lateral movement as:
```
EventID=4625 Account=CORP\alice FailureReason=Unknown user or bad password IpAddress=10.0.2.33
```
A **CEF syslog** from a Palo Alto NGFW looks like:
```
CEF:0|Palo Alto Networks|PAN-OS|10.0|TRAFFIC|deny|7|src=10.0.2.55 dst=8.8.8.8 dpt=53
```

Without normalization, correlation across these three sources is impossible.
The parser maps all three into a **Common Information Model (CIM)** — one schema,
vendor-neutral field names.

| Raw Field | Cisco ASA | Windows Event | CEF | CIM Field |
|-----------|-----------|---------------|-----|-----------|
| Source IP | `src outside:X/port` | `IpAddress=X` | `src=X` | `src_ip` |
| Dest IP | `dst inside:X/port` | (host field) | `dst=X` | `dst_ip` |
| Action | message ID 106023 = Deny | EventID 4625 = Fail | `act=deny` | `action` |
| User | (not applicable) | `Account=X` | `suser=X` | `user` |

*Network analogy: log normalization is what OSPF does for routing metrics — it converts
heterogeneous per-protocol costs into one comparable metric so you can make routing decisions
across vendors. CIM converts heterogeneous log formats into one comparable schema so you can
correlate across sources.*

In [ ]:
# ── Demo 1: Multi-Source Log Parser ───────────────────────────────────────────

@dataclass
class NormalizedEvent:
    """Common Information Model (CIM) — vendor-neutral log event schema."""
    timestamp:   str
    source_type: str          # cisco_asa | windows_event | cef
    src_ip:      str = ""
    dst_ip:      str = ""
    src_port:    int = 0
    dst_port:    int = 0
    protocol:    str = ""
    action:      str = ""     # allow | deny | fail | success
    user:        str = ""
    event_id:    str = ""
    severity:    int = 5      # 1=critical, 10=informational (syslog-like)
    raw:         str = ""


# ── Cisco ASA parser ──────────────────────────────────────────────────────────
# Example: %ASA-4-106023: Deny tcp src outside:185.220.101.47/52111 dst inside:10.0.1.5/443
ASA_PATTERN = re.compile(
    r"%ASA-(?P<sev>\d)-(?P<msgid>\d+):?\s+"
    r"(?P<action>\w+)\s+(?P<proto>\w+)\s+"
    r"src\s+\w+:(?P<src_ip>[\d.]+)/(?P<src_port>\d+)\s+"
    r"dst\s+\w+:(?P<dst_ip>[\d.]+)/(?P<dst_port>\d+)",
    re.IGNORECASE
)

def parse_cisco_asa(raw: str, ts: str) -> Optional[NormalizedEvent]:
    m = ASA_PATTERN.search(raw)
    if not m:
        return None
    action_raw = m.group("action").lower()
    return NormalizedEvent(
        timestamp=ts, source_type="cisco_asa",
        src_ip=m.group("src_ip"), dst_ip=m.group("dst_ip"),
        src_port=int(m.group("src_port")), dst_port=int(m.group("dst_port")),
        protocol=m.group("proto").lower(),
        action="deny" if "deny" in action_raw else "allow",
        event_id=m.group("msgid"), severity=int(m.group("sev")),
        raw=raw
    )


# ── Windows Security Event Log parser ─────────────────────────────────────────
# Example: EventID=4625 Account=CORP\alice IpAddress=10.0.2.33 FailureReason=Bad password
WIN_EVENT_ID_MAP = {
    "4624": "success", "4625": "fail", "4648": "explicit_cred",
    "4720": "account_created", "4728": "group_member_added",
}

def parse_windows_event(raw: str, ts: str) -> Optional[NormalizedEvent]:
    eid = re.search(r"EventID=(\d+)", raw)
    ip  = re.search(r"IpAddress=([\d.]+)", raw)
    user = re.search(r"Account=([\S]+)", raw)
    if not eid:
        return None
    action = WIN_EVENT_ID_MAP.get(eid.group(1), "unknown")
    return NormalizedEvent(
        timestamp=ts, source_type="windows_event",
        src_ip=ip.group(1) if ip else "",
        user=user.group(1) if user else "",
        action=action,
        event_id=eid.group(1),
        severity=3 if action == "fail" else 7,
        raw=raw
    )


# ── CEF (Common Event Format) parser ─────────────────────────────────────────
# Example: CEF:0|Palo Alto|PAN-OS|10.0|TRAFFIC|deny|7|src=10.0.2.55 dst=8.8.8.8 dpt=53
def parse_cef(raw: str, ts: str) -> Optional[NormalizedEvent]:
    if not raw.startswith("CEF:"):
        return None
    # Header: CEF:version|vendor|product|version|sig_id|name|severity|extension
    parts = raw.split("|", 7)
    if len(parts) < 8:
        return None
    name     = parts[5].strip().lower()
    ext      = parts[7]
    src_ip   = re.search(r"src=([\d.]+)", ext)
    dst_ip   = re.search(r"dst=([\d.]+)", ext)
    dst_port = re.search(r"dpt=(\d+)",    ext)
    src_port = re.search(r"spt=(\d+)",    ext)
    proto    = re.search(r"proto=(\w+)",   ext)
    user     = re.search(r"suser=(\S+)",   ext)
    return NormalizedEvent(
        timestamp=ts, source_type="cef",
        src_ip=src_ip.group(1) if src_ip else "",
        dst_ip=dst_ip.group(1) if dst_ip else "",
        src_port=int(src_port.group(1)) if src_port else 0,
        dst_port=int(dst_port.group(1)) if dst_port else 0,
        protocol=proto.group(1).lower() if proto else "",
        action="deny" if "deny" in name or "block" in name else "allow",
        user=user.group(1) if user else "",
        event_id=parts[4].strip(),
        severity=int(parts[6].strip()) if parts[6].strip().isdigit() else 5,
        raw=raw
    )


def parse_log(raw: str, ts: str) -> Optional[NormalizedEvent]:
    """Auto-detect log format and dispatch to correct parser."""
    if raw.startswith("%ASA"):
        return parse_cisco_asa(raw, ts)
    if raw.startswith("CEF:"):
        return parse_cef(raw, ts)
    if "EventID=" in raw:
        return parse_windows_event(raw, ts)
    return None


# ── Sample raw logs from three different sources ───────────────────────────────
RAW_LOGS = [
    ("2024-01-15T03:02:11Z",
     "%ASA-4-106023: Deny tcp src outside:185.220.101.47/52111 dst inside:10.0.1.5/443 by access-group outside_in"),
    ("2024-01-15T03:02:18Z",
     "%ASA-4-106023: Deny tcp src outside:185.220.101.47/52112 dst inside:10.0.1.5/22 by access-group outside_in"),
    ("2024-01-15T03:02:33Z",
     "%ASA-6-302013: Built inbound TCP connection 1234 for outside:10.0.1.10/55002 to inside:10.0.1.5/80"),
    ("2024-01-15T03:05:44Z",
     "EventID=4625 Account=CORP\\alice IpAddress=185.220.101.47 FailureReason=Unknown user or bad password WorkstationName=UNKNOWN"),
    ("2024-01-15T03:05:47Z",
     "EventID=4625 Account=CORP\\alice IpAddress=185.220.101.47 FailureReason=Unknown user or bad password WorkstationName=UNKNOWN"),
    ("2024-01-15T03:05:51Z",
     "EventID=4625 Account=CORP\\administrator IpAddress=185.220.101.47 FailureReason=Unknown user or bad password WorkstationName=UNKNOWN"),
    ("2024-01-15T03:07:12Z",
     "CEF:0|Palo Alto Networks|PAN-OS|10.1|TRAFFIC|deny|7|src=185.220.101.47 dst=10.0.1.5 dpt=3389 spt=61234 proto=tcp"),
    ("2024-01-15T03:07:45Z",
     "CEF:0|Palo Alto Networks|PAN-OS|10.1|TRAFFIC|allow|3|src=10.0.1.22 dst=8.8.8.8 dpt=53 spt=54321 proto=udp suser=alice"),
]

print("=" * 70)
print("MULTI-SOURCE LOG PARSER — NORMALIZATION TO CIM SCHEMA")
print("=" * 70)
print(f"{'Time':25} {'Source':14} {'Action':8} {'Src IP':18} {'Dst IP':16} {'DPort':6} {'User'}")
print("-" * 100)

normalized_events: List[NormalizedEvent] = []
parse_stats = defaultdict(int)

for ts, raw in RAW_LOGS:
    ev = parse_log(raw, ts)
    if ev:
        normalized_events.append(ev)
        parse_stats[ev.source_type] += 1
        user_col = ev.user[:12] if ev.user else "-"
        print(f"{ev.timestamp:25} {ev.source_type:14} {ev.action:8} "
              f"{ev.src_ip:18} {ev.dst_ip:16} {ev.dst_port:<6} {user_col}")
    else:
        print(f"[PARSE FAIL] {raw[:60]}...")

print(f"\nParsed {len(normalized_events)}/{len(RAW_LOGS)} events")
print("Source breakdown:", dict(parse_stats))
print("\nAll events now share the same CIM schema — ready for correlation.")

## Demo 2: Pattern-Based Threat Detector

Once logs are normalized, you can run **detection rules** that work across all source types.
Three classic attack patterns that appear in firewall + auth log combinations:

| Pattern | What to Look For | Time Window | Threshold |
|---------|-----------------|-------------|----------|
| **Brute Force** | N failed auth attempts from same IP | 5 minutes | ≥ 5 failures |
| **Port Scan** | Same IP connecting to N distinct destination ports | 2 minutes | ≥ 10 ports |
| **Firewall Deny Storm** | Sudden spike in deny events from external IP | 1 minute | ≥ 20 denies |

These are **rule-based correlation** — deterministic, auditable, fast.
No ML required. The LLM adds the context layer: *why* is this suspicious,
*what MITRE technique* does it represent, *what should the analyst do first*.

```
185.220.101.47 timeline (last 6 minutes):
  03:02:11  ASA deny tcp -> 10.0.1.5:443
  03:02:18  ASA deny tcp -> 10.0.1.5:22
  03:05:44  Windows 4625 (auth fail) for CORP\alice
  03:05:47  Windows 4625 (auth fail) for CORP\alice
  03:05:51  Windows 4625 (auth fail) for CORP\administrator
  03:07:12  CEF deny tcp -> 10.0.1.5:3389
  Detection: BRUTE_FORCE + PORT_SCAN + DENY_STORM from same external IP
```

*Network analogy: this is your SNMP trap correlation logic — you don't alert on one
interface flap. You alert when three interfaces on the same upstream device flap within
30 seconds. Pattern recognition across events, not individual events.*

In [ ]:
# ── Demo 2: Pattern-Based Threat Detector ─────────────────────────────────────

from collections import defaultdict
from datetime import datetime

@dataclass
class ThreatSignal:
    pattern:     str
    src_ip:      str
    count:       int
    window_sec:  int
    severity:    str
    details:     dict


def parse_ts(ts_str: str) -> float:
    """Parse ISO timestamp to Unix float for window arithmetic."""
    return datetime.strptime(ts_str, "%Y-%m-%dT%H:%M:%SZ").timestamp()


def detect_brute_force(
    events: List[NormalizedEvent],
    window_sec: int = 300,
    threshold: int = 5
) -> List[ThreatSignal]:
    """
    Detect brute force: N failed auth events from same source IP in window.
    Looks at windows_event and ASA auth failures.
    """
    # Group failed auth events by src_ip
    failures: Dict[str, List[float]] = defaultdict(list)
    for ev in events:
        is_auth_fail = (
            (ev.source_type == "windows_event" and ev.action == "fail") or
            (ev.source_type == "cisco_asa" and ev.event_id in ("106023", "113005", "113006"))
        )
        if is_auth_fail and ev.src_ip:
            failures[ev.src_ip].append(parse_ts(ev.timestamp))

    signals = []
    for ip, times in failures.items():
        times.sort()
        # Sliding window: find any window of `window_sec` with >= threshold events
        for i in range(len(times)):
            window = [t for t in times if times[i] <= t <= times[i] + window_sec]
            if len(window) >= threshold:
                signals.append(ThreatSignal(
                    pattern="BRUTE_FORCE",
                    src_ip=ip,
                    count=len(window),
                    window_sec=window_sec,
                    severity="HIGH",
                    details={"fail_count": len(window), "window_minutes": window_sec // 60}
                ))
                break  # one alert per IP
    return signals


def detect_port_scan(
    events: List[NormalizedEvent],
    window_sec: int = 120,
    threshold: int = 5
) -> List[ThreatSignal]:
    """
    Detect port scan: same src IP connecting to N distinct dst_ports in window.
    Looks at firewall deny events (scanning typically hits closed ports).
    """
    ip_ports: Dict[str, Dict[float, set]] = defaultdict(lambda: defaultdict(set))
    for ev in events:
        if ev.action == "deny" and ev.src_ip and ev.dst_port > 0:
            t = parse_ts(ev.timestamp)
            ip_ports[ev.src_ip][t].add(ev.dst_port)

    signals = []
    for ip, time_ports in ip_ports.items():
        times = sorted(time_ports.keys())
        for i in range(len(times)):
            window_ports: set = set()
            for t in times:
                if times[i] <= t <= times[i] + window_sec:
                    window_ports.update(time_ports[t])
            if len(window_ports) >= threshold:
                signals.append(ThreatSignal(
                    pattern="PORT_SCAN",
                    src_ip=ip,
                    count=len(window_ports),
                    window_sec=window_sec,
                    severity="HIGH",
                    details={"distinct_ports": sorted(window_ports), "window_seconds": window_sec}
                ))
                break
    return signals


def detect_deny_storm(
    events: List[NormalizedEvent],
    window_sec: int = 60,
    threshold: int = 3
) -> List[ThreatSignal]:
    """
    Detect deny storm: N deny events from same src IP in short window.
    Threshold lowered for demo data set size.
    """
    deny_times: Dict[str, List[float]] = defaultdict(list)
    for ev in events:
        if ev.action == "deny" and ev.src_ip:
            deny_times[ev.src_ip].append(parse_ts(ev.timestamp))

    signals = []
    for ip, times in deny_times.items():
        times.sort()
        for i in range(len(times)):
            window = [t for t in times if times[i] <= t <= times[i] + window_sec]
            if len(window) >= threshold:
                signals.append(ThreatSignal(
                    pattern="DENY_STORM",
                    src_ip=ip,
                    count=len(window),
                    window_sec=window_sec,
                    severity="MEDIUM",
                    details={"deny_count": len(window), "window_seconds": window_sec}
                ))
                break
    return signals


# ── Run all detectors on the normalized events ────────────────────────────────
print("=" * 60)
print("PATTERN-BASED THREAT DETECTOR")
print("=" * 60)

all_signals: List[ThreatSignal] = []
all_signals.extend(detect_brute_force(normalized_events))
all_signals.extend(detect_port_scan(normalized_events))
all_signals.extend(detect_deny_storm(normalized_events))

if not all_signals:
    print("No patterns detected in current event set.")
else:
    for sig in all_signals:
        print(f"\n[{sig.severity}] {sig.pattern}")
        print(f"  Source IP : {sig.src_ip}")
        print(f"  Count     : {sig.count} events in {sig.window_sec}s window")
        print(f"  Details   : {sig.details}")

        # LLM triage for each signal
        analysis = llm_analyze(
            f"Security pattern detected: {sig.pattern} from {sig.src_ip}. "
            f"Details: {sig.details}. "
            f"Which MITRE ATT&CK technique? What is the immediate recommended action? "
            f"Keep response under 70 words.",
            max_tokens=100
        )
        print(f"  LLM       : {analysis}")

print(f"\n[Detector] Total signals generated: {len(all_signals)}")

## Demo 3: Log Correlation Engine

Individual signals from one source are weak evidence. **Correlated signals** across
multiple sources from the same attacker IP, within the same time window,
are strong evidence.

The correlation engine answers: **"What else happened around the same time as this event,
involving the same IP or user?"**

```
Correlation key:  IP = 185.220.101.47  |  Time window: ±10 minutes

  03:02:11  [cisco_asa]     DENY tcp -> 10.0.1.5:443
  03:02:18  [cisco_asa]     DENY tcp -> 10.0.1.5:22
  03:05:44  [windows_event] FAIL auth CORP\alice
  03:05:47  [windows_event] FAIL auth CORP\alice
  03:05:51  [windows_event] FAIL auth CORP\administrator
  03:07:12  [cef]           DENY tcp -> 10.0.1.5:3389

  -> Correlated incident: Multi-stage attack from single external IP
     Phase 1 (03:02): Reconnaissance / port scanning
     Phase 2 (03:05): Credential attack against discovered services
     Phase 3 (03:07): Continued port scan including RDP
```

This is **session windowing** — grouping causally-related events into an incident context.
Each session gets a single LLM analysis instead of per-event alerts, dramatically
reducing analyst workload.

In [ ]:
# ── Demo 3: Log Correlation Engine ────────────────────────────────────────────

@dataclass
class CorrelatedIncident:
    """A group of related events linked by IP and time window."""
    incident_id: str
    pivot_ip:    str
    events:      List[NormalizedEvent]
    signals:     List[ThreatSignal]
    start_time:  str
    end_time:    str
    source_types: List[str]


def correlate_by_ip(
    events:     List[NormalizedEvent],
    signals:    List[ThreatSignal],
    window_sec: int = 600
) -> List[CorrelatedIncident]:
    """
    Group events and signals by src_ip within a time window.
    Returns one CorrelatedIncident per unique attacker IP that has >= 1 signal.
    """
    # Find IPs that have at least one signal
    signal_ips = {s.src_ip for s in signals}

    incidents = []
    for ip in signal_ips:
        # Gather all events involving this IP (as src or referenced in user context)
        related_events = [
            ev for ev in events
            if ev.src_ip == ip
        ]
        if not related_events:
            continue

        related_events.sort(key=lambda e: e.timestamp)
        ip_signals = [s for s in signals if s.src_ip == ip]

        incidents.append(CorrelatedIncident(
            incident_id=f"INC-{hash(ip) % 9000 + 1000:04d}",
            pivot_ip=ip,
            events=related_events,
            signals=ip_signals,
            start_time=related_events[0].timestamp,
            end_time=related_events[-1].timestamp,
            source_types=list({ev.source_type for ev in related_events})
        ))

    return incidents


def format_incident_for_llm(inc: CorrelatedIncident) -> str:
    """Render correlated incident as structured text for LLM analysis."""
    lines = [
        f"Incident {inc.incident_id} | Attacker IP: {inc.pivot_ip}",
        f"Timespan: {inc.start_time} to {inc.end_time}",
        f"Sources: {', '.join(inc.source_types)}",
        f"Patterns detected: {', '.join(s.pattern for s in inc.signals)}",
        "",
        "Event timeline:"
    ]
    for ev in inc.events:
        dst_info = f"-> {ev.dst_ip}:{ev.dst_port}" if ev.dst_ip else ""
        user_info = f"user={ev.user}" if ev.user else ""
        lines.append(
            f"  {ev.timestamp[11:19]} [{ev.source_type}] {ev.action} "
            f"{dst_info} {user_info}"
        )
    return "\n".join(lines)


# ── Run correlation ────────────────────────────────────────────────────────────
print("=" * 60)
print("LOG CORRELATION ENGINE — MULTI-SOURCE INCIDENT GROUPING")
print("=" * 60)

incidents = correlate_by_ip(normalized_events, all_signals)

if not incidents:
    print("No correlated incidents found.")
else:
    for inc in incidents:
        print(f"\n{'='*55}")
        print(f"INCIDENT: {inc.incident_id}")
        print(f"Pivot IP: {inc.pivot_ip}")
        print(f"Events  : {len(inc.events)} across {len(inc.source_types)} source types")
        print(f"Signals : {[s.pattern for s in inc.signals]}")
        print(f"Window  : {inc.start_time} -> {inc.end_time}")

        print("\nTimeline:")
        for ev in inc.events:
            dst_info  = f"-> {ev.dst_ip}:{ev.dst_port}" if ev.dst_ip else ""
            user_info = f"[user: {ev.user}]" if ev.user else ""
            print(f"  {ev.timestamp[11:19]} [{ev.source_type:14}] "
                  f"{ev.action:7} {dst_info:22} {user_info}")

        # LLM full incident analysis
        incident_text = format_incident_for_llm(inc)
        llm_prompt = (
            f"{incident_text}\n\n"
            f"Analyze this multi-source correlated incident. "
            f"What attack phases are visible? MITRE ATT&CK techniques? "
            f"What is the likely attacker objective? Top 3 immediate analyst actions. "
            f"Under 120 words."
        )
        analysis = llm_analyze(llm_prompt, max_tokens=180)
        print(f"\nLLM Incident Analysis:\n  {analysis}")

print(f"\n[Correlation] {len(incidents)} incident(s) assembled from {len(all_signals)} signals.")

## Demo 4: SIEM Alert Enrichment with Threat Intel

Raw alerts contain the *what* (IP, port, action). **Enrichment** adds the *so what*:
is this IP known-bad? Which country? Which ASN? Has it appeared in threat feeds?

Real SIEM enrichment queries live APIs (VirusTotal, Shodan, MaxMind GeoIP, AbuseIPDB).
This demo simulates a **local threat intelligence cache** — the same logic, runnable offline.

| Enrichment Layer | Data Source | What It Adds |
|-----------------|-------------|-------------|
| **Geo lookup** | MaxMind GeoIP / ip-api.com | Country, ASN, org name |
| **Reputation score** | AbuseIPDB / VirusTotal | Abuse confidence % |
| **Threat feed match** | Emerging Threats / Feodo Tracker | Known malware C2, TOR exit node |
| **BGP context** | BGPView / Team Cymru | Hosting ASN, announced prefix |

After enrichment, the analyst can immediately see:
```
185.220.101.47:
  Country   : Germany (DE)
  ASN       : AS24940 Hetzner Online (VPS hosting)
  Abuse %   : 94% (known scanner)
  Threat    : TOR Exit Node (Feodo Tracker)
  Verdict   : KNOWN_BAD — escalate immediately
```

That context changes an analyst's response time from "need to research" to
"block and escalate" — a 10-minute task becoming a 30-second decision.

In [ ]:
# ── Demo 4: SIEM Alert Enrichment with Threat Intel ───────────────────────────

@dataclass
class ThreatIntelEntry:
    ip:           str
    country:      str
    country_code: str
    asn:          str
    org:          str
    abuse_score:  int          # 0-100 (AbuseIPDB-style)
    threat_tags:  List[str]    # e.g. ["tor_exit", "scanner", "c2_host"]
    last_seen:    str


# Simulated threat intel cache
# In production: query AbuseIPDB, MaxMind, VirusTotal, Feodo Tracker
THREAT_INTEL_DB: Dict[str, ThreatIntelEntry] = {
    "185.220.101.47": ThreatIntelEntry(
        ip="185.220.101.47", country="Germany", country_code="DE",
        asn="AS24940", org="Hetzner Online GmbH",
        abuse_score=94,
        threat_tags=["tor_exit_node", "known_scanner", "brute_force_source"],
        last_seen="2024-01-14T23:55:00Z"
    ),
    "91.108.4.12": ThreatIntelEntry(
        ip="91.108.4.12", country="Russia", country_code="RU",
        asn="AS42831", org="UK-2 Limited",
        abuse_score=78,
        threat_tags=["c2_infrastructure", "cobalt_strike"],
        last_seen="2024-01-15T01:22:00Z"
    ),
    "203.0.113.99": ThreatIntelEntry(
        ip="203.0.113.99", country="China", country_code="CN",
        asn="AS4134", org="CHINANET-BACKBONE",
        abuse_score=61,
        threat_tags=["proxy", "vpn_exit"],
        last_seen="2024-01-10T18:30:00Z"
    ),
    "10.0.1.22": ThreatIntelEntry(
        ip="10.0.1.22", country="Internal", country_code="--",
        asn="RFC1918", org="Internal Network",
        abuse_score=0,
        threat_tags=[],
        last_seen=""
    ),
}


def enrich_ip(ip: str) -> Optional[ThreatIntelEntry]:
    """Look up IP in threat intel cache. Returns None if not found (unknown IP)."""
    return THREAT_INTEL_DB.get(ip)


def verdict(entry: ThreatIntelEntry) -> str:
    """Compute verdict from enrichment data."""
    if not entry or entry.abuse_score == 0:
        return "CLEAN"
    if entry.abuse_score >= 80 or "c2_infrastructure" in entry.threat_tags:
        return "KNOWN_BAD"
    if entry.abuse_score >= 50 or any(t in entry.threat_tags for t in ["tor_exit_node", "scanner"]):
        return "SUSPICIOUS"
    return "UNKNOWN"


@dataclass
class EnrichedAlert:
    incident_id:  str
    src_ip:       str
    signals:      List[str]
    intel:        Optional[ThreatIntelEntry]
    verdict:      str
    llm_summary:  str = ""


def enrich_incident(inc: CorrelatedIncident) -> EnrichedAlert:
    intel  = enrich_ip(inc.pivot_ip)
    verd   = verdict(intel) if intel else "UNKNOWN"
    signals = [s.pattern for s in inc.signals]
    return EnrichedAlert(
        incident_id=inc.incident_id,
        src_ip=inc.pivot_ip,
        signals=signals,
        intel=intel,
        verdict=verd
    )


# ── Run enrichment on all incidents ───────────────────────────────────────────
print("=" * 60)
print("SIEM ALERT ENRICHMENT — THREAT INTEL OVERLAY")
print("=" * 60)

enriched_alerts: List[EnrichedAlert] = []

for inc in incidents:
    alert = enrich_incident(inc)
    enriched_alerts.append(alert)

    print(f"\n{'='*55}")
    print(f"Incident : {alert.incident_id}")
    print(f"Src IP   : {alert.src_ip}")
    print(f"Signals  : {alert.signals}")
    print(f"Verdict  : {alert.verdict}")

    if alert.intel:
        i = alert.intel
        print(f"\nThreat Intel:")
        print(f"  Country    : {i.country} ({i.country_code})")
        print(f"  ASN / Org  : {i.asn} — {i.org}")
        print(f"  Abuse Score: {i.abuse_score}%")
        print(f"  Tags       : {i.threat_tags}")
        print(f"  Last Seen  : {i.last_seen}")
    else:
        print("  [No threat intel available for this IP]")

    # LLM generates enriched analyst briefing
    intel_str = (
        f"country={alert.intel.country}, ASN={alert.intel.asn} ({alert.intel.org}), "
        f"abuse={alert.intel.abuse_score}%, tags={alert.intel.threat_tags}"
        if alert.intel else "No intel available"
    )
    llm_prompt = (
        f"SIEM enriched alert — verdict: {alert.verdict}\n"
        f"IP: {alert.src_ip}\n"
        f"Patterns: {alert.signals}\n"
        f"Threat intel: {intel_str}\n\n"
        f"Generate a 3-sentence analyst briefing: "
        f"what happened, why the intel makes this credible, recommended next step."
    )
    summary = llm_analyze(llm_prompt, max_tokens=120)
    alert.llm_summary = summary
    print(f"\nAnalyst Briefing:\n  {summary}")

print(f"\n[Enrichment] {len(enriched_alerts)} alerts enriched.")

## Demo 5: Full Log Analysis Pipeline

All four stages wired together into a complete **log analysis pipeline** —
the way a production AI-augmented SIEM operates:

```
Stage 1: INGEST
  Raw logs (ASA syslog, Windows events, CEF) -> queue

Stage 2: PARSE & NORMALIZE
  Multi-source parser -> CIM NormalizedEvent objects

Stage 3: DETECT
  Pattern-based rules -> ThreatSignal list

Stage 4: CORRELATE
  IP + time-window grouping -> CorrelatedIncident list

Stage 5: ENRICH
  Threat intel overlay -> EnrichedAlert list

Stage 6: LLM SUMMARY
  claude-sonnet: prioritized incident report for analyst handoff
```

The analyst receives a prioritized, pre-enriched, pre-correlated report.
Their job is not "gather context" — it is **"make the decision."**

> **The 1,000:1 ratio of generated-to-analyzed logs collapses when the pipeline
> does the triage automatically. The analyst sees 3 incidents, not 10,000 events.**

**Non-negotiable guardrail:**
> All recommended containment actions require explicit analyst approval.
> The pipeline enriches and prioritizes. The analyst decides.
> Automated blocking without human review causes outages.

In [ ]:
# ── Demo 5: Full Log Analysis Pipeline ────────────────────────────────────────

class SIEMPipeline:
    """
    Full 6-stage log analysis pipeline:
    Ingest -> Parse -> Detect -> Correlate -> Enrich -> LLM Summary
    """

    def __init__(self):
        self.raw_queue:     List[tuple]              = []
        self.normalized:    List[NormalizedEvent]     = []
        self.signals:       List[ThreatSignal]        = []
        self.incidents:     List[CorrelatedIncident]  = []
        self.enriched:      List[EnrichedAlert]       = []
        self.stats:         Dict[str, int]            = defaultdict(int)

    # Stage 1: Ingest
    def ingest(self, raw_logs: List[tuple]):
        self.raw_queue.extend(raw_logs)
        self.stats["ingested"] += len(raw_logs)
        print(f"[Stage 1: INGEST]    {len(raw_logs)} raw log lines queued")

    # Stage 2: Parse & Normalize
    def normalize(self):
        parsed, failed = 0, 0
        for ts, raw in self.raw_queue:
            ev = parse_log(raw, ts)
            if ev:
                self.normalized.append(ev)
                parsed += 1
            else:
                failed += 1
        self.stats["parsed"]  = parsed
        self.stats["dropped"] = failed
        print(f"[Stage 2: NORMALIZE] {parsed} events normalized, {failed} dropped")

    # Stage 3: Detect
    def detect(self):
        self.signals.extend(detect_brute_force(self.normalized))
        self.signals.extend(detect_port_scan(self.normalized))
        self.signals.extend(detect_deny_storm(self.normalized))
        self.stats["signals"] = len(self.signals)
        print(f"[Stage 3: DETECT]    {len(self.signals)} threat signals generated")

    # Stage 4: Correlate
    def correlate(self):
        self.incidents = correlate_by_ip(self.normalized, self.signals)
        self.stats["incidents"] = len(self.incidents)
        print(f"[Stage 4: CORRELATE] {len(self.incidents)} correlated incident(s)")

    # Stage 5: Enrich
    def enrich(self):
        for inc in self.incidents:
            self.enriched.append(enrich_incident(inc))
        self.stats["enriched"] = len(self.enriched)
        print(f"[Stage 5: ENRICH]    {len(self.enriched)} alerts enriched with threat intel")

    # Stage 6: LLM Analyst Summary
    def generate_report(self) -> str:
        """Generate final analyst handoff report via LLM (Sonnet for quality)."""
        if not self.enriched:
            return "No incidents to report."

        incident_summaries = []
        for a in self.enriched:
            intel = (
                f"{a.intel.country} / {a.intel.org} / abuse={a.intel.abuse_score}% / tags={a.intel.threat_tags}"
                if a.intel else "unknown"
            )
            incident_summaries.append(
                f"Incident {a.incident_id}: {a.src_ip} | patterns={a.signals} | "
                f"verdict={a.verdict} | intel=({intel})"
            )

        prompt = (
            "You are a senior security analyst. "
            "Write a concise shift handoff report for the following SIEM incidents.\n"
            "For each incident: priority (CRITICAL/HIGH/MEDIUM), "
            "plain-language summary of what happened, "
            "MITRE ATT&CK phase, "
            "and the single most important analyst action.\n\n"
            "INCIDENTS:\n" + "\n".join(incident_summaries) + "\n\n"
            "Keep total response under 200 words. Use bullet points."
        )

        if USE_API:
            resp = client.messages.create(
                model="claude-sonnet-4-5-20250514",
                max_tokens=300,
                messages=[{"role": "user", "content": prompt}]
            )
            return resp.content[0].text.strip()
        return (
            "[SIMULATION MODE — sample report]\n"
            "CRITICAL: INC-XXXX — 185.220.101.47 (TOR exit node, DE, Hetzner) "
            "conducted multi-stage attack: port scan + brute force + deny storm. "
            "MITRE: T1595 (Reconnaissance) -> T1110 (Brute Force). "
            "ACTION: Block IP at perimeter ASA + Palo Alto, preserve auth logs for "
            "forensic review, verify no successful authentications occurred."
        )

    def run_pipeline(self, raw_logs: List[tuple]):
        """Execute all 6 stages sequentially."""
        print("=" * 60)
        print("SIEM PIPELINE — FULL RUN")
        print("=" * 60)
        self.ingest(raw_logs)
        self.normalize()
        self.detect()
        self.correlate()
        self.enrich()

        print("\n[Stage 6: LLM REPORT] Generating analyst handoff...")
        report = self.generate_report()

        print("\n" + "=" * 60)
        print("ANALYST HANDOFF REPORT")
        print("=" * 60)
        print(report)

        print("\n" + "=" * 60)
        print("PIPELINE STATISTICS")
        print("=" * 60)
        for stage, count in self.stats.items():
            reduction = ""
            if stage == "signals" and self.stats["parsed"]:
                pct = 100 * count / self.stats["parsed"]
                reduction = f" ({pct:.1f}% of events produced a signal)"
            if stage == "incidents" and self.stats["signals"]:
                pct = 100 * count / self.stats["signals"]
                reduction = f" ({pct:.0f}% of signals correlated into incidents)"
            print(f"  {stage:12}: {count}{reduction}")

        print("\n[GUARDRAIL] All recommended actions await analyst approval.")
        print("[GUARDRAIL] No automated blocking executed. Human decision required.")


# ── Extended raw log set for pipeline demo ────────────────────────────────────
EXTENDED_LOGS = RAW_LOGS + [
    ("2024-01-15T03:08:00Z",
     "%ASA-4-106023: Deny tcp src outside:185.220.101.47/52200 dst inside:10.0.1.5/8080 by access-group outside_in"),
    ("2024-01-15T03:08:05Z",
     "%ASA-4-106023: Deny tcp src outside:185.220.101.47/52201 dst inside:10.0.1.5/8443 by access-group outside_in"),
    ("2024-01-15T03:09:00Z",
     "EventID=4625 Account=CORP\\svc-backup IpAddress=185.220.101.47 FailureReason=Bad password WorkstationName=UNKNOWN"),
    ("2024-01-15T03:09:20Z",
     "EventID=4625 Account=CORP\\svc-deploy IpAddress=185.220.101.47 FailureReason=Bad password WorkstationName=UNKNOWN"),
    ("2024-01-15T03:10:00Z",
     "CEF:0|Juniper|SRX|12.1|TRAFFIC|deny|7|src=185.220.101.47 dst=10.0.2.1 dpt=161 spt=54999 proto=udp"),
]

pipeline = SIEMPipeline()
pipeline.run_pipeline(EXTENDED_LOGS)

## Summary: What You Built

You now have a working **Security Log Analysis & SIEM Pipeline** with 5 stages:

| Stage | Component | Input | Output |
|-------|-----------|-------|--------|
| **1. Parse** | Multi-source parser | Raw syslog / Win events / CEF | NormalizedEvent (CIM schema) |
| **2. Detect** | Rule engine | NormalizedEvent stream | ThreatSignal list |
| **3. Correlate** | Time-window grouper | Events + Signals | CorrelatedIncident |
| **4. Enrich** | Threat intel lookup | Incident + IP | EnrichedAlert + verdict |
| **5. Report** | LLM Sonnet | EnrichedAlert list | Analyst handoff report |

### Why This Architecture Works

- **No single log source is sufficient** — attack phases span firewall, auth, DNS, proxy
- **Normalization before correlation** — CIM schema makes cross-vendor detection possible
- **Rules for the known, LLM for context** — deterministic detection + natural language explanation
- **Session windowing** collapses 10,000 events into 3 analyst-ready incidents

### Production Upgrade Path

```
Regex parsers              ->  Logstash / Elastic Agent / Cribl pipelines
In-memory threat intel DB  ->  Live AbuseIPDB + VirusTotal + MaxMind API
Simple time-window grouper ->  Graph-based correlation (Neo4j + Cypher)
claude-haiku-4-5 triage    ->  claude-sonnet-4-5 for incident reports
Flat list alert queue      ->  SOAR platform integration (Palo Alto XSOAR)
```

### The Non-Negotiable Rule

> **The pipeline enriches and prioritizes. The analyst decides.**
> Every containment recommendation — blocking an IP, disabling an account,
> isolating a host — requires explicit human approval before execution.
> Alert fatigue is defeated by better triage, not by automated blocking.

In [ ]:
# ── Chapter 67: Production Deployment Checklist ────────────────────────────────
print("CHAPTER 67: PRODUCTION DEPLOYMENT CHECKLIST")
print("=" * 60)

checklist = [
    # Log sources
    ("Log Sources",     "Cisco ASA / Juniper SRX syslog (UDP 514 or TCP 6514 TLS)"),
    ("Log Sources",     "Windows Event Log via WEF or Winlogbeat -> Kafka"),
    ("Log Sources",     "Palo Alto / Fortinet NGFWs in CEF format"),
    ("Log Sources",     "RADIUS / Tacacs+ auth logs + DNS resolver query logs"),
    # Normalization
    ("Normalization",   "Pick one CIM: Elastic Common Schema (ECS) or OCSF"),
    ("Normalization",   "Write parser for EVERY source before writing detection rules"),
    ("Normalization",   "Validate: same event from ASA and Palo Alto -> identical CIM fields"),
    # Detection tuning
    ("Detection",       "Brute force: tune threshold per service (SSH: 5, web: 20)"),
    ("Detection",       "Port scan: separate internal/external scan thresholds"),
    ("Detection",       "Measure false positive rate — target < 15% for rule-based"),
    # Correlation
    ("Correlation",     "Correlation window: 10min for most attacks, 24h for slow burns"),
    ("Correlation",     "Multiple pivot dimensions: IP + user + hostname + file hash"),
    ("Correlation",     "MITRE ATT&CK kill chain mapping for each correlation pattern"),
    # LLM integration
    ("LLM",             "claude-haiku-4-5-20251001: per-signal triage (fast + cheap)"),
    ("LLM",             "claude-sonnet-4-5-20250514: incident reports (quality matters)"),
    ("LLM",             "Always include raw evidence in prompt — no hallucinated context"),
    # Guardrails
    ("Guardrails",      "Human approval required before any automated blocking action"),
    ("Guardrails",      "Immutable audit log: every LLM recommendation + analyst decision"),
    ("Guardrails",      "Alert fatigue metric: track alerts-generated vs alerts-investigated"),
]

current_cat = ""
for cat, item in checklist:
    if cat != current_cat:
        print(f"\n[{cat}]")
        current_cat = cat
    print(f"  + {item}")

print("\n" + "=" * 60)
print("KEY FORMULAS")
print("=" * 60)
print("Alert fatigue ratio   = alerts_fired / alerts_investigated  (target: < 5x)")
print("Detection precision   = true_positives / (true_positives + false_positives)")
print("Pipeline compression  = raw_events / analyst_incidents  (10,000:3 = 3,333x)")
print("\nAll 5 stages run in sequence. Output of each feeds the next.")
print("The analyst sees incidents, not events. That is the entire value proposition.")
print("\nChapter 67 complete. Build the pipeline. Normalize first. Correlate always.")